# Probe: teacher-anchored residual MixPL-style DQA-MoX

- created_utc: 2026-05-11T14:49:16+00:00
- target_mAP50: 0.600
- workspace: `/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27d_probe_teacher_residual_mixpl_r2`
- log: `/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27d_probe_teacher_residual_mixpl_r2/logs/27d_probe_teacher_residual_mixpl_r2_train.log`
- research_note: `/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/reports/27_research_note_iter_000_27d_probe_teacher_residual_mixpl_r2.md`

## Current Results

| trial | best mAP50 | mAP50:95 |
|---|---:|---:|
| 24a_client_dominant_soft_expand | 0.457000 | 0.258000 |
| 27a_soft_mixture_head_first_40_10 |  |  |
| 27b_localization_uncertainty_strict_then_open |  |  |
| 27b_probe_localization_uncertainty_r2 | 0.462000 | 0.260000 |
| 27c_probe_k6_night_tail_r2 | 0.459000 | 0.250000 |

## Hypothesis

27cのK=6 night-tail化はhighway_nightとtotalを同時に悪化させた。MixPLはpseudo-labelが検出器の弱点を増幅し、tail/small objectのmissを悪化させると指摘している。そこでMoE容量を増やすのではなく、warmup teacherを強くanchorし、client更新を小さいresidual/adaptorとしてだけ混ぜる。sourceを多めに反復し、pseudo box lossを極小にして、pseudoGTは主にobjectness/router/domain信号として使う2-round probeにする。

## Paper Basis

- FedMoX/PSSFL: https://arxiv.org/abs/2508.16568
  FedMoX treats the practical setting as server labeled high-resolution data plus client unlabeled low-resolution data, and uses sparse MoE with a spatial router and Soft-Mixture to stabilize semi-supervised FL.
- PseCo: https://arxiv.org/abs/2203.16317
  PseCo argues that classification score alone does not guarantee localization precision; prediction-guided assignment and consistency voting make learning robust to coarse boxes.
- Rethinking Pseudo Labels: https://arxiv.org/abs/2106.00168
  Certainty-aware pseudo labels combine classification and localization quality, dynamically adjust thresholds, and reweight category losses to reduce class imbalance.
- Mixed Pseudo Labels: https://arxiv.org/abs/2312.07006
  MixPL shows that pseudo labels can amplify both a detector's strengths and weaknesses, especially missed detections for small and tail-category objects; this supports keeping pseudoGT as a weak residual/domain signal when recent probes drift below warmup.


In [1]:
import csv
import json
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path('/app/Object_Detection')
WORKSPACE = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27d_probe_teacher_residual_mixpl_r2')
LOG_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27d_probe_teacher_residual_mixpl_r2/logs/27d_probe_teacher_residual_mixpl_r2_train.log')
METRICS_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27d_probe_teacher_residual_mixpl_r2/stats/18_client_balanced_single_injection_dqamox_final_metrics.csv')
CMD = ['/opt/venv/bin/python', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/scripts/run_scene_daynight_dqa_18_client_balanced_single_injection_dqamox.py', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27d_probe_teacher_residual_mixpl_r2', '--repair-baseline-rounds', '0', '--source-workspace', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/08_full_latent_dqamox_from_warmup', '--source-repair-baseline-rounds', '30', '--target-map50', '0.6', '--num-experts', '4', '--top-k', '2', '--router-temperature', '1.1', '--router-balance-weight', '0.018', '--router-entropy-weight', '0.0004', '--dqa-client-balance-stats', '--dqa-client-balance-target', 'median', '--dqa-client-balance-max-scale', '4.0', '--load-bias-strength', '0.25', '--batch-size', '80', '--workers', '8', '--gpus', '2', '--max-images-per-client', '0', '--master-port', '39000', '--evaluate', '--classwise', '--no-eval-plots', '--force', '--warmup-epochs', '50', '--client-limit', '1200', '--client-sampling-ratio', '0.500', '--client-sampling-seed', '270309', '--phase1-rounds', '2', '--phase2-rounds', '0', '--phase1-train-scope', 'neck_head', '--phase1-repair-train-scope', 'neck_head', '--phase1-client-epochs', '1', '--phase1-client-lr', '0.00016', '--phase1-source-repeat', '3', '--phase1-pseudo-repeat', '2', '--phase1-loss-box', '0.00012', '--phase2-train-scope', 'all', '--phase2-repair-train-scope', 'all', '--phase2-client-epochs', '1', '--phase2-client-lr', '0.00003', '--phase2-source-repeat', '2', '--phase2-pseudo-repeat', '1', '--phase2-loss-box', '0.00003', '--server-repair-epochs', '1', '--server-repair-lr', '0.00010', '--server-repair-loss-box', '0.003', '--dqa-temperature', '0.95', '--dqa-uniform-mix', '0.18', '--dqa-classwise-blend', '0.18', '--dqa-stability-lambda', '0.45', '--dqa-server-anchor', '0.72', '--dqa-min-server-alpha', '0.68', '--dqa-residual-blend', '0.04', '--late-dqa-server-anchor', '0.66', '--late-dqa-min-server-alpha', '0.62', '--late-dqa-residual-blend', '0.04', '--curriculum-start-round', '3', '--expert-keep-fraction', '0.88', '--expert-max-class-fraction', '0.34', '--actual-max-class-fraction', '0.46', '--late-expert-keep-fraction', '0.92', '--late-expert-max-class-fraction', '0.38', '--late-actual-max-class-fraction', '0.50', '--min-score', '0.14', '--min-stability', '0.42', '--late-min-score', '0.12', '--late-min-stability', '0.36', '--max-boxes-per-image', '16', '--skip-warmup-training', '--warmup-checkpoint', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/08_full_latent_dqamox_from_warmup/checkpoints/round000_latent_dqamox_warmup.pt']

WORKSPACE.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
(WORKSPACE / "stats").mkdir(parents=True, exist_ok=True)
(WORKSPACE / "stats" / "27_notebook_command.json").write_text(
    json.dumps({"command": CMD}, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print(" ".join(CMD))
with LOG_PATH.open("w", encoding="utf-8") as log:
    proc = subprocess.run(CMD, cwd=REPO_ROOT, stdout=log, stderr=subprocess.STDOUT, check=False)
print("returncode", proc.returncode)
print("log", LOG_PATH)
if proc.returncode != 0:
    raise SystemExit(proc.returncode)


/opt/venv/bin/python /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/scripts/run_scene_daynight_dqa_18_client_balanced_single_injection_dqamox.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27d_probe_teacher_residual_mixpl_r2 --repair-baseline-rounds 0 --source-workspace /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/08_full_latent_dqamox_from_warmup --source-repair-baseline-rounds 30 --target-map50 0.6 --num-experts 4 --top-k 2 --router-temperature 1.1 --router-balance-weight 0.018 --router-entropy-weight 0.0004 --dqa-client-balance-stats --dqa-client-balance-target median --dqa-client-balance-max-scale 4.0 --load-bias-strength 0.25 --batch-size 80 --workers 8 --gpus 2 --max-images-per-client 0 --master-port 39000 --evaluate --classwise --no-eval-plots --force --warmup-epochs 50 --client-limi

returncode 0
log /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27d_probe_teacher_residual_mixpl_r2/logs/27d_probe_teacher_residual_mixpl_r2_train.log


In [2]:
import csv
from pathlib import Path

METRICS_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27d_probe_teacher_residual_mixpl_r2/stats/18_client_balanced_single_injection_dqamox_final_metrics.csv')
rows = list(csv.DictReader(METRICS_PATH.open(encoding="utf-8"))) if METRICS_PATH.exists() else []
for row in rows:
    print(row)


{'checkpoint_label': 'warmup_global', 'condition': 'warmup', 'kind': 'warmup', 'phase': '', 'round': '', 'precision': '0.681', 'recall': '0.427', 'map50': '0.460000', 'map50_95': '0.259000', 'gain_vs_warmup_map50_95': '0.000000', 'delta_vs_server_repair_map50_95': '0.058000', 'worst_split': 'highway_night', 'worst_split_map50_95': '0.173000', 'day_avg_map50_95': '0.283667', 'night_avg_map50_95': '0.203333', 'day_night_gap_map50_95': '0.080333'}
{'checkpoint_label': 'warmup_server_repair_final', 'condition': 'warmup + server repair', 'kind': 'server_repair', 'phase': '0', 'round': '30', 'precision': '0.662', 'recall': '0.366', 'map50': '0.378000', 'map50_95': '0.201000', 'gain_vs_warmup_map50_95': '-0.058000', 'delta_vs_server_repair_map50_95': '0.000000', 'worst_split': 'highway_night', 'worst_split_map50_95': '0.134000', 'day_avg_map50_95': '0.226000', 'night_avg_map50_95': '0.148667', 'day_night_gap_map50_95': '0.077333'}
{'checkpoint_label': 'latent_dqamox_final_aggregate', 'conditi